## [Cell 1] 환경 설정 및 LLM 초기화

In [13]:
# 필요한 기본 패키지 로드 및 LLM 설정
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# .env 파일에서 환경 변수(API 키) 로드
load_dotenv()

# LangChain 0.3.x 방식의 모델 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

### [Cell 2] 문제 2-1 : 콤마 구분 리스트 파서 활용

In [14]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 1. 파서 생성 및 포맷 지시사항 추출
comma_parser = CommaSeparatedListOutputParser()
format_instructions = comma_parser.get_format_instructions()

# 2. 프롬프트 템플릿 생성
prompt = PromptTemplate(
    template="사용자가 관심있는 분야와 관련된 한국의 유명한 장소나 활동 5가지를 추천해주세요.\n\n{format_instructions}\n\n관심 분야: {subject}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions}
)

# 3. 체인 구성 (LCEL)
chain = prompt | llm | comma_parser

# 4. 실행 및 결과 확인
result = chain.invoke({"subject": "음식"})
print("=== 문제 2-1 결과 ===")
print(result)

=== 문제 2-1 결과 ===
['광장시장', '남대문시장', '인사동 전통찻집', '홍대 푸드트럭', '부산 자갈치시장']


### [Cell 3] 문제 2-2 : 영화 리뷰 감정 분석기

In [18]:
from enum import Enum
from pydantic import BaseModel, Field

# 1. Enum 클래스에 중립(NEUTRAL) 값 추가
class Sentiment(Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL = "neutral"  # 보통/중립 추가

# 2. Pydantic 스키마 정의
class ReviewAnalysis(BaseModel):
    sentiment: Sentiment = Field(
        description="리뷰의 감정 (positive: 긍정, negative: 부정, neutral: 보통/중립)"
    )

# 3. 모델에 구조화 출력 적용
structured_llm = llm.with_structured_output(ReviewAnalysis)

# 4. 테스트 리뷰 검증
reviews = [
    "이 영화 정말 재미없어요. 시간 낭비였습니다.",
    "배우들의 연기가 훌륭하고 스토리도 감동적이었어요!",
    "그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요."
]

print("=== 문제 2-2 결과 ===")
for review in reviews:
    res = structured_llm.invoke(f"다음 영화 리뷰를 읽고 감정을 분류해주세요.\n\n리뷰: {review}")
    print(f"리뷰: {review}")
    print(f"분류: {res.sentiment.value}\n")



=== 문제 2-2 결과 ===
리뷰: 이 영화 정말 재미없어요. 시간 낭비였습니다.
분류: negative

리뷰: 배우들의 연기가 훌륭하고 스토리도 감동적이었어요!
분류: positive

리뷰: 그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.
분류: neutral



### [Cell 4] 문제 2-3 : 학생 정보 구조화 시스템

In [16]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# 1. Pydantic을 이용한 데이터 모델 정의
class StudentInfo(BaseModel):
    name: str = Field(description="학생의 이름")
    age: int = Field(description="학생의 나이")
    major: str = Field(description="학생의 전공")
    hobbies: List[str] = Field(description="학생의 취미 리스트")
    goal: str = Field(description="학생의 목표")

# 2. 파서 및 포맷 지시사항 생성
pydantic_parser = PydanticOutputParser(pydantic_object=StudentInfo)
format_instructions = pydantic_parser.get_format_instructions()

# 3. 프롬프트 템플릿 작성
prompt = PromptTemplate(
    template="다음 텍스트에서 학생 정보를 구조화된 형태로 추출해주세요.\n\n{format_instructions}\n\n텍스트: {text}",
    input_variables=["text"],
    partial_variables={"format_instructions": format_instructions}
)

# 4. 체인 구성
chain = prompt | llm | pydantic_parser

# 5. 실행
sample_text = "안녕하세요! 저는 김민수이고 22살입니다. 컴퓨터공학을 전공하고 있어요. 취미로는 게임하기, 영화보기, 코딩을 좋아합니다. 앞으로 훌륭한 개발자가 되는 것이 목표입니다."
result = chain.invoke({"text": sample_text})

print("=== 문제 2-3 결과 ===")
# dict 형태로 출력하거나 Pydantic의 model_dump_json() 활용
print(result.model_dump_json(indent=4))

=== 문제 2-3 결과 ===
{
    "name": "김민수",
    "age": 22,
    "major": "컴퓨터공학",
    "hobbies": [
        "게임하기",
        "영화보기",
        "코딩"
    ],
    "goal": "앞으로 훌륭한 개발자가 되는 것"
}


### [Cell 5] 문제 2-4 : 여행 계획 분석기

In [17]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema 

# 1. ResponseSchema 정의
response_schemas = [
    ResponseSchema(name="destination", description="여행지"),
    ResponseSchema(name="duration", description="여행 기간"),
    ResponseSchema(name="budget", description="여행 예산"),
    ResponseSchema(name="rating", description="추천도 (1~5점)"),
    ResponseSchema(name="activities", description="주요 활동 리스트", type="list")
]

# 2. 파서 및 포맷 지시사항 생성
structured_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = structured_parser.get_format_instructions()

# 3. 프롬프트 템플릿 생성
prompt = PromptTemplate(
    template="다음 텍스트에서 여행 관련 핵심 정보를 추출해주세요.\n\n{format_instructions}\n\n텍스트: {text}",
    input_variables=["text"],
    partial_variables={"format_instructions": format_instructions}
)

# 4. 체인 구성
chain = prompt | llm | structured_parser

# 5. 실행
travel_text = "지난 주에 부산으로 2박 3일 여행을 다녀왔어요. 총 30만원 정도 썼는데 해운대에서 바다구경하고, 자갈치시장에서 회 먹고, 감천문화마을도 구경했어요. 정말 만족스러운 여행이었습니다. 5점 만점에 4점 정도 줄 수 있을 것 같아요."
result = chain.invoke({"text": travel_text})

print("=== 문제 2-4 결과 ===")
import json
print(json.dumps(result, indent=4, ensure_ascii=False))

=== 문제 2-4 결과 ===
{
    "destination": "부산",
    "duration": "2박 3일",
    "budget": "30만원",
    "rating": "4",
    "activities": [
        "해운대에서 바다구경",
        "자갈치시장에서 회 먹기",
        "감천문화마을 구경"
    ]
}


In [35]:
from enum import Enum
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

# ==========================================
# 1. LLM 초기화 (원격 Ollama 서버 연동 설정)
# ==========================================
# 필요에 따라 모델명과 base_url을 환경에 맞게 조정하세요.
llm = ChatOllama(
    model="qwen2.5:0.5b",  # 또는 qwen3 등 사용 중인 모델명
    base_url="http://192.168.45.218:11434"
)


# ==========================================
# 2. Pydantic 스키마 및 Enum 정의
# ==========================================

# [문제 2-1 대응] 추천 리스트 형태
class RecommendationList(BaseModel):
    items: List[str] = Field(description="추천 항목 리스트 5개")


# [문제 2-2 대응] 감정 분석 (Enum 활용)
class Sentiment(Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL = "neutral"  # 보통/중립


class ReviewAnalysis(BaseModel):
    sentiment: Sentiment = Field(
        description="리뷰 감정 분류 (positive, negative, neutral)"
    )


# [문제 2-3 대응] 학생 정보 추출
class StudentInfo(BaseModel):
    name: str = Field(description="학생 이름")
    age: int = Field(description="나이")
    major: str = Field(description="전공")
    hobbies: List[str] = Field(description="취미 리스트")
    goal: str = Field(description="목표")


# [문제 2-4 대응] 여행 정보 추출
class TravelInfo(BaseModel):
    destination: str = Field(description="여행지")
    duration: str = Field(description="여행 기간")
    budget: str = Field(description="여행 예산")
    rating: int = Field(description="추천도 (1~5점)")
    activities: List[str] = Field(description="주요 활동 리스트")


# ==========================================
# 3. 통합 실행 함수 정의
# ==========================================
def analyze_with_schema(
    llm_instance,
    pydantic_schema: BaseModel,
    system_prompt: str,
    user_prompt: str,
    **kwargs,
):
    """시스템 프롬프트, 사용자 프롬프트 템플릿, 그리고 가변 인자(kwargs)를 동적으로 처리하는 통합 함수"""
    structured_llm = llm_instance.with_structured_output(pydantic_schema)

    prompt = ChatPromptTemplate.from_messages(
        [("system", system_prompt), ("human", user_prompt)]
    )

    chain = prompt | structured_llm
    return chain.invoke(kwargs)


# ==========================================
# 4. 각 문제별 통합 테스트 실행
# ==========================================

print("=== [테스트 2-1] 장소/활동 추천 5가지 ===")
res_2_1 = analyze_with_schema(
    llm_instance=llm,
    pydantic_schema=RecommendationList,
    system_prompt="사용자의 관심 분야와 관련된 한국의 유명한 장소나 활동 5가지를 추천해주세요.",
    user_prompt="관심 분야: {subject}",
    subject="음식",
)
print(res_2_1)
print()


print("=== [테스트 2-2] 영화 리뷰 감정 분석 (Neutral 포함) ===")
reviews = [
    "이 영화 정말 재미없어요. 시간 낭비였습니다.",
    "배우들의 연기가 훌륭하고 스토리도 감동적이었어요!",
    "그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.",
]
for review in reviews:
    res_2_2 = analyze_with_schema(
        llm_instance=llm,
        pydantic_schema=ReviewAnalysis,
        system_prompt="다음 영화 리뷰를 읽고 감정을 분류해주세요.",
        user_prompt="리뷰: {review}",
        review=review,
    )
    print(f"리뷰: {review}")
    print(f"분류: {res_2_2.sentiment.value}\n")


print("=== [테스트 2-3] 학생 정보 구조화 추출 ===")
sample_text = "안녕하세요! 저는 김민수이고 22살입니다. 컴퓨터공학을 전공하고 있어요. 취미로는 게임하기, 영화보기, 코딩을 좋아합니다. 앞으로 훌륭한 개발자가 되는 것이 목표입니다."
res_2_3 = analyze_with_schema(
    llm_instance=llm,
    pydantic_schema=StudentInfo,
    system_prompt="다음 텍스트에서 학생 정보를 구조화된 형태로 정확히 추출해주세요.",
    user_prompt="텍스트: {text}",
    text=sample_text,
)
print(res_2_3.model_dump_json(indent=4))
print()


print("=== [테스트 2-4] 여행 정보 구조화 추출 ===")
travel_text = "지난 주에 부산으로 2박 3일 여행을 다녀왔어요. 총 30만원 정도 썼는데 해운대에서 바다구경하고, 자갈치시장에서 회 먹고, 감천문화마을도 구경했어요. 정말 만족스러운 여행이었습니다. 5점 만점에 4점 정도 줄 수 있을 것 같아요."
res_2_4 = analyze_with_schema(
    llm_instance=llm,
    pydantic_schema=TravelInfo,
    system_prompt="다음 텍스트에서 여행 관련 핵심 정보를 추출해주세요.",
    user_prompt="텍스트: {text}",
    text=travel_text,
)
print(res_2_4.model_dump_json(indent=4))

=== [테스트 2-1] 장소/활동 추천 5가지 ===
items=['네트워크', '마트', '음료', '음식점', '네트워크', '식당', '카페', '전통식품', '유명식품', '음료', '음식점', '카페', '유명식품', '음식점', '맛집', '유명식품', '네트워크', '카페', '음식점', '맛집', '음료', '맛집', '카페', '네트워크', '유명식품']

=== [테스트 2-2] 영화 리뷰 감정 분석 (Neutral 포함) ===
리뷰: 이 영화 정말 재미없어요. 시간 낭비였습니다.
분류: negative

리뷰: 배우들의 연기가 훌륭하고 스토리도 감동적이었어요!
분류: positive

리뷰: 그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.
분류: neutral

=== [테스트 2-3] 학생 정보 구조화 추출 ===
{
    "name": "김민수",
    "age": 22,
    "major": "컴퓨터공학",
    "hobbies": [
        "게임하기",
        "영화보기",
        "코딩"
    ],
    "goal": "향후 훌륭한 개발자가 되기"
}

=== [테스트 2-4] 여행 정보 구조화 추출 ===
{
    "destination": "부산",
    "duration": "3일",
    "budget": "30만원",
    "rating": 5,
    "activities": [
        "해운대에서 바다구경하고",
        "자갈치시장에서 회먹고",
        "감천문화마을도 구경"
    ]
}
